# Practice run of analysing/testing different models on the UNSW_NB15 dataset, before trying Deep Learning.

Prior research suggests this is a largely non-linear, less separable dataset so deep learning may be necessary, but I will try simpler, more interpretable models first for the sake of completeness, and to gain Variable Importances

Let's load our packages and data

In [7]:
#import packages:
from google.colab import drive

try:
  import google.colab
  IN_COLAB = True
except:
  IN_COLAB = False

if IN_COLAB:
  # Check if drive is mounted by looking for the mount point in the file system.
  # This is a more robust approach than relying on potentially internal variables.
  import os
  if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')
  else:
    print("Google Drive is already mounted.")
else:
  print("Not running in Google Colab. Drive mounting skipped.")

import pandas as pd
import sklearn as sk
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from tqdm import tqdm







print("New run: Packages loaded")

Google Drive is already mounted.
New run: Packages loaded


In [2]:
#if using colabs - will need to first mount your drive

#change these for different users
test_set_filepath = '/content/drive/MyDrive/Colab_Notebooks/Data/UNSW_NB15_testing-set.parquet'
training_set_filepath = '/content/drive/MyDrive/Colab_Notebooks/Data/UNSW_NB15_training-set.parquet'

# Import the two CSV files
test_set = pd.read_parquet(test_set_filepath)
train_set = pd.read_parquet(training_set_filepath)



The next cell does some basic analysis, and one hot encodes some of the features:

In [3]:
"""
#print number of records in our data
print(f"Number of records in training set: {len(train_set)}")
print(f"Number of records in test set: {len(test_set)}")

#lets see which ones are categorical etc
print(f'''
The columns and datatypes are:
{train_set.dtypes}
''')

print("Categorical Columns are :", categorical_cols)

#print out number of classifications
print(f"Number of categories in 'label' category: {len(train_set['label'].unique())}")

#print out labels
print(f"Labels: {train_set['label'].unique()}")

#print out how many unique values we have for each categorical variable - if we have too many we may need an embeddings layer
for col in categorical_cols:
    print(f"Number of categories in '{col}' category: {len(train_set[col].unique())}")

"""



'\n#print number of records in our data\nprint(f"Number of records in training set: {len(train_set)}")\nprint(f"Number of records in test set: {len(test_set)}")\n\n#lets see which ones are categorical etc\nprint(f\'\'\'\nThe columns and datatypes are:\n{train_set.dtypes}\n\'\'\')\n\nprint("Categorical Columns are :", categorical_cols)\n\n#print out number of classifications\nprint(f"Number of categories in \'label\' category: {len(train_set[\'label\'].unique())}")\n\n#print out labels\nprint(f"Labels: {train_set[\'label\'].unique()}")\n\n#print out how many unique values we have for each categorical variable - if we have too many we may need an embeddings layer\nfor col in categorical_cols:\n    print(f"Number of categories in \'{col}\' category: {len(train_set[col].unique())}")\n\n'

In [4]:

def preprocess_data(data_set):
  """
  Function to preprocess data. One hot encodes the top 6 most common values for 'proto'. NOTE - to self, might need to do this grouping for other categoricals?
  And turns boolean columns into 1s and 0s.

  Args:
  data_set (dataframe) : test or train set to be processed

  Retuns:
  data_set (dataframe) : processed data set

  """

  #we aer going to set this as a binary classification task for now. This is because Logistic Regression struggles with multiclass, and these interpretable models are only EDA really
  if 'attack_cat' in data_set.columns.tolist():
    data_set.drop('attack_cat', axis = 1, inplace = True)

  #there seems to be over 100 possible values of proto - lets see how common they all are
  if 'proto' in data_set.columns.tolist():
    category_percentages = data_set['proto'].value_counts(normalize=True) * 100

    #define a dict of the categories and their percentages of occurence. what we want to do here is group any that occur less than 0.5% of the time, into an 'other' category
    category_percentages_dict = category_percentages.to_dict()
    #we can then print this to view the distributions ^

    # After looking at the distributions of the possible values for Proto, only the top 6 occur more than 0.5% of the time - hence all others are very rare
    # So we get the top 6 most common values. We have to hardcode in this value of top 6
    top_6_categories = category_percentages.head(6).index.tolist()

    #we now have a list of values that we want to one hot encode. we want to simply group the others into an 'other column'
    data_set['proto_grouped'] = data_set['proto'].apply(lambda x: x if x in top_6_categories else 'other')

    #now we one hot encode this column
    data_set = pd.get_dummies(data_set, columns=['proto_grouped'])

    #drop the original columns if still present
    data_set = data_set.drop('proto', axis=1)

  if 'proto_grouped' in data_set.columns.tolist():
      data_set = data_set.drop(['proto_grouped'], axis=1)

  # List only the categorical columns (object types)
  categorical_cols = data_set.select_dtypes(include=['category']).columns.tolist()

  #one hot encode all remaining categorical columns
  data_set = pd.get_dummies(data_set, columns=categorical_cols)

  #encode all binary data as 1s and 0s
  binary_cols = data_set.select_dtypes(include=['bool']).columns

  #convert to int - the original apply function was returning a dataframe instead of a series in cases where more than 1 value was found. This is because the column has more than 1 unique value. This can happen when for example, you expect all the values to be `True` but then find some are `False` as well. in that scenario the apply function will not collapse this down to a series.
  data_set[binary_cols] = data_set[binary_cols].apply(lambda x: x.astype(int))

  print(f"Data set preprocessed, columns = {data_set.columns.tolist()}")

  return data_set

train_set = preprocess_data(train_set)




Data set preprocessed, columns = ['dur', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit', 'swin', 'stcpb', 'dtcpb', 'dwin', 'tcprtt', 'synack', 'ackdat', 'smean', 'dmean', 'trans_depth', 'response_body_len', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd', 'is_sm_ips_ports', 'label', 'proto_grouped_arp', 'proto_grouped_ospf', 'proto_grouped_other', 'proto_grouped_sctp', 'proto_grouped_tcp', 'proto_grouped_udp', 'proto_grouped_unas', 'service_-', 'service_dhcp', 'service_dns', 'service_ftp', 'service_ftp-data', 'service_http', 'service_irc', 'service_pop3', 'service_radius', 'service_smtp', 'service_snmp', 'service_ssh', 'service_ssl', 'state_CON', 'state_ECO', 'state_FIN', 'state_INT', 'state_PAR', 'state_REQ', 'state_RST', 'state_URN', 'state_no']


[dtype('float32') dtype('int16') dtype('int32') dtype('int64')
 dtype('int8')]


NOTE TO SELF -
1. THIS IS FOR BINARY CLASSIFICATION, WE WANT MULTICLASS EVENTUALLY, BUT FOR NOW WE WILL JUST DO BN


Based on the high number of columns in the Proto column, we may want to consider an Embeddings layer with the Deep Learning that we plan to undertake later. However since DT/RF perform somewhat poorly on sparse vector datasets (like one hot encoded ones) we will group all the extremely rare categories into an 'other'.


In [5]:
def run_models(model_type, test_set = test_set, train_set = train_set):
  """
  Runs LR, DT or RF model on dataframe
  """

  #drop label and define list of out targets
  X_train = train_set.drop('label', axis=1)
  y_train = train_set['label']

  #do the same for test set
  X_test = test_set.drop('label', axis=1)
  y_test = test_set['label']

  # List only the categorical columns (object types)
  categorical_cols = train_set.select_dtypes(include=['category']).columns.tolist()

  param_grid_dt = {
    'max_depth': [3, 5, 10],             # List of max depth values
    'min_samples_split': [2, 10, 20],    # List of minimum samples to split
    'min_samples_leaf': [1, 5, 10]       # List of minimum samples per leaf
  }

  param_grid_rf = {
    'n_estimators': [50, 100, 200],      # List of number of trees to use
    'max_depth': [5, 10, 20],            # List of maximum depths for trees
    'min_samples_split': [2, 10, 20]     # List of minimum samples for splitting a node
  }


#=============================== LR ========================================#

  if model_type.upper() == 'LR':

    # Define the pipeline
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(max_iter=1000))
    ])

    # Define the hyperparameter grid
    param_grid = {
        'classifier__C': [0.01, 0.1, 1, 10, 100],
        'classifier__penalty': ['l1', 'l2'],
        'classifier__solver': ['liblinear']
    }

    # Set up cross-validation strategies
    # Set up cross-validation strategies
    inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    outer_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

    print("KFolds defined, beginning nested cross-validation.")

    # Outer loop (on training data)
    outer_scores = []
    for train_index, val_index in tqdm(outer_cv.split(X_train, y_train)):  # Split training data
        X_train_fold, X_val_fold = X_train.iloc[train_index], X_train.iloc[val_index]
        y_train_fold, y_val_fold = y_train.iloc[train_index], y_train.iloc[val_index]

        # Set up GridSearchCV with the pipeline (inner loop)
        grid_search = GridSearchCV(
            estimator=pipeline,  # Your pipeline object
            param_grid=param_grid,  # Your parameter grid
            cv=inner_cv,  # Use the defined inner_cv
            scoring='roc_auc',
            n_jobs=-1
        )

        # Fit GridSearchCV on the training fold
        grid_search.fit(X_train_fold, y_train_fold)

        # Evaluate on the validation fold
        score = grid_search.score(X_val_fold, y_val_fold)
        outer_scores.append(score)

    # Print the average outer score (ROC AUC) on training data
    print(f"Average Validation ROC AUC: {np.mean(outer_scores)}")

    # Evaluate the best model on the held-out test set
    best_model = grid_search.best_estimator_
    test_score = best_model.score(X_test, y_test)
    print(f"Test ROC AUC: {test_score}")


#=============================== DT ========================================#

  if model_type.upper() == 'DT':

    pass
#=============================== RF ========================================#

  if model_type.upper() == 'RF':

    pass

run_models('LR')

KFolds defined, beginning nested cross-validation.


0it [01:39, ?it/s]


KeyboardInterrupt: 